## Generate Random Zoom Outs change the path after processing train or val
#### Create two folders named TRAIN_ZOOM_OUT_CLS and TEST_ZOOM_OUT_CLS

In [2]:
import os
import random
from PIL import Image
from collections import defaultdict

def apply_zoom_out(image_path, zoom_out_factor, output_image_path):
    """Apply zoom out augmentation by adding white borders."""
    original_image = Image.open(image_path)
    img_width, img_height = original_image.size

    # Calculate new image dimensions
    new_width = int(img_width / zoom_out_factor)
    new_height = int(img_height / zoom_out_factor)

    # Create a white background image
    zoomed_out_image = Image.new('RGB', (new_width, new_height), color='white')

    # Position original image in the center
    left = (new_width - img_width) // 2
    top = (new_height - img_height) // 2
    zoomed_out_image.paste(original_image, (left, top))

    # Resize back to original size (optional)
    zoomed_out_image = zoomed_out_image.resize((img_width, img_height), Image.LANCZOS)

    # Save the zoomed-out image
    zoomed_out_image.save(output_image_path)
    return True

# CHANGE TARGET COUNT -----------------------------------
def balance_dataset(base_path, target_count=175, checkpoint=10):
    """Balance dataset by applying zoom-out augmentation."""
    zoom_out_factors = [round(0.88 - x * 0.04, 2) for x in range(7)]  # [0.88, 0.84, ...]

    # Output directory for augmented data
    # CHANGE THIS ---------------------------------------------------
    output_base_path = r"F:\SNAPFOLIA\TRAIN_ZOOM_OUT_CLS"
    os.makedirs(output_base_path, exist_ok=True)

    # Process each class directory
    class_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

    for class_dir in class_dirs:
        class_path = os.path.join(base_path, class_dir)

        # Get original images directly inside the class folder
        original_images = [f for f in os.listdir(class_path)
                           if f.lower().endswith(('.jpg', '.jpeg', '.png')) and '(z' not in f]

        if not original_images:
            print(f"❗ Skipping {class_dir}: No valid images found.")
            continue

        print(f"\n📂 Processing {class_dir}: {len(original_images)} original images found.")
        print(f"🔄 Generating {target_count} augmented images...")

        # Tracking zoom-out factors for each image
        used_zoom_factors = defaultdict(set)

        augmentations_created = 0
        attempts = 0
        max_attempts = target_count * 40  # More attempts for better diversity

        while augmentations_created < target_count and attempts < max_attempts:
            attempts += 1
            image_file = random.choice(original_images)

            # Available zoom-out factors
            available_factors = [f for f in zoom_out_factors
                                 if f not in used_zoom_factors[image_file]]

            if not available_factors:
                continue  # Skip if no zoom-out factors left

            zoom_out_factor = random.choice(available_factors)
            used_zoom_factors[image_file].add(zoom_out_factor)

            # Image paths
            image_path = os.path.join(class_path, image_file)
            base_name, ext = os.path.splitext(image_file)
            new_image_name = f"{base_name}(z{zoom_out_factor}){ext}"
            output_image_path = os.path.join(output_base_path, class_dir, new_image_name)

            os.makedirs(os.path.dirname(output_image_path), exist_ok=True)

            if apply_zoom_out(image_path, zoom_out_factor, output_image_path):
                augmentations_created += 1

                # Display checkpoints for progress
                if augmentations_created % checkpoint == 0:
                    print(f"✅ Generated {augmentations_created}/{target_count} images...")

        print(f"\n✅ Completed augmentation for {class_dir}")
        print(f"📊 Total augmented images created: {augmentations_created}")
        print("\n🔎 Zoom-out factors used per image:")
        for img, factors in used_zoom_factors.items():
            if factors:
                print(f"  {img}: {sorted(factors)}")

if __name__ == "__main__":
    dataset_path = r"F:\SNAPFOLIA\CLS_DS_59 - SPLIT\train"  # CHANGE THIS TO DATASET PATH
    print("🚀 Starting dataset augmentation...")
    balance_dataset(dataset_path)
    print("\n🎯 Dataset augmentation completed!")

🚀 Starting dataset augmentation...

📂 Processing Acacia: 256 original images found.
🔄 Generating 175 augmented images...
✅ Generated 10/175 images...
✅ Generated 20/175 images...
✅ Generated 30/175 images...
✅ Generated 40/175 images...
✅ Generated 50/175 images...
✅ Generated 60/175 images...
✅ Generated 70/175 images...
✅ Generated 80/175 images...
✅ Generated 90/175 images...
✅ Generated 100/175 images...
✅ Generated 110/175 images...
✅ Generated 120/175 images...
✅ Generated 130/175 images...
✅ Generated 140/175 images...
✅ Generated 150/175 images...
✅ Generated 160/175 images...
✅ Generated 170/175 images...

✅ Completed augmentation for Acacia
📊 Total augmented images created: 175

🔎 Zoom-out factors used per image:
  env_Acacia-J-31-_jpg.rf.0b87913e01652177558f56c8d6c85b38.jpg: [0.76, 0.88]
  white_bg_Acacia-Z-104-_jpg.rf.52af85c5ca1f0083e214224db5e6ef7a.jpg: [0.64, 0.76, 0.8, 0.88]
  env_Acacia-K-6-_jpg.rf.8496e4c040f06d2fb14c0761e4133981.jpg: [0.68, 0.72, 0.8]
  env_Acacia-H-

## Generate Random Zoom In change the path after processing train or val
#### No need to create folders

In [4]:
import os
import random
from PIL import Image
from collections import defaultdict

class ZoomAugmentor:
    def __init__(self, base_path, target_count=350):
        self.base_path = base_path
        self.target_count = target_count
        self.zoom_factors = [round(x * 0.02 + 1.04, 2) for x in range(7)]  # [1.04, 1.06, ..., 1.16]

    def apply_zoom(self, image_path, zoom_factor, output_image_path):
        """Apply zoom augmentation."""
        image = Image.open(image_path)
        img_width, img_height = image.size

        # Crop dimensions for zoom
        crop_width, crop_height = img_width / zoom_factor, img_height / zoom_factor
        left, top = (img_width - crop_width) / 2, (img_height - crop_height) / 2
        right, bottom = left + crop_width, top + crop_height

        # Crop and resize the image
        cropped_image = image.crop((left, top, right, bottom))
        zoomed_image = cropped_image.resize((img_width, img_height), Image.LANCZOS)

        # Save the zoomed image
        zoomed_image.save(output_image_path)

    def balance_dataset(self):
        """Balance dataset by adding zoomed augmentations."""
        class_dirs = [d for d in os.listdir(self.base_path) 
                      if os.path.isdir(os.path.join(self.base_path, d))]

        for class_dir in class_dirs:
            class_path = os.path.join(self.base_path, class_dir)

            # Original images
            original_images = [f for f in os.listdir(class_path)
                               if f.endswith(('.jpg', '.jpeg', '.png')) and '(z' not in f]

            print(f"\nProcessing {class_dir}:")
            print(f"Original images: {len(original_images)}")

            current_count = len(original_images) + len([f for f in os.listdir(class_path) if '(z' in f])
            num_needed = max(0, self.target_count - current_count)

            print(f"Current total images: {current_count}")
            print(f"Additional images needed: {num_needed}")

            if num_needed == 0:
                print("No augmentation needed for this class")
                continue

            used_zoom_factors = defaultdict(set)
            max_possible = len(original_images) * len(self.zoom_factors)
            if max_possible < num_needed:
                print(f"Warning: Can only generate {max_possible} unique augmentations")
                num_needed = max_possible

            augmentations_created = 0
            attempts = 0
            max_attempts = num_needed * 20  # Increased attempts for retries

            while augmentations_created < num_needed and attempts < max_attempts:
                attempts += 1
                image_file = random.choice(original_images)

                # Available zoom factors for this image
                available_factors = [f for f in self.zoom_factors 
                                     if f not in used_zoom_factors[image_file]]

                if not available_factors:
                    continue  # Skip if no available zoom factors

                zoom_factor = random.choice(available_factors)
                used_zoom_factors[image_file].add(zoom_factor)

                # Path setup
                image_path = os.path.join(class_path, image_file)
                base_name = os.path.splitext(image_file)[0]
                ext = os.path.splitext(image_file)[1]
                new_image_name = f"{base_name}(z{zoom_factor}){ext}"

                output_image_path = os.path.join(class_path, new_image_name)

                self.apply_zoom(image_path, zoom_factor, output_image_path)
                augmentations_created += 1

                if augmentations_created % 10 == 0:
                    print(f"Generated {augmentations_created}/{num_needed} augmented images")

            print(f"\nCompleted augmentation for {class_dir}")
            print(f"Total augmented images created: {augmentations_created}")
            print("\nZoom factors used per image:")
            for img, factors in used_zoom_factors.items():
                if factors:
                    print(f"  {img}: {sorted(factors)}")

if __name__ == "__main__":
    dataset_path = r"F:\SNAPFOLIA\CLS_DS_59 - SPLIT\train"

    print("Starting dataset balancing...")
    augmentor = ZoomAugmentor(dataset_path)
    augmentor.balance_dataset()
    print("\nDataset balancing completed!")

Starting dataset balancing...

Processing Acacia:
Original images: 256
Current total images: 256
Additional images needed: 94
Generated 10/94 augmented images
Generated 20/94 augmented images
Generated 30/94 augmented images
Generated 40/94 augmented images
Generated 50/94 augmented images
Generated 60/94 augmented images
Generated 70/94 augmented images
Generated 80/94 augmented images
Generated 90/94 augmented images

Completed augmentation for Acacia
Total augmented images created: 94

Zoom factors used per image:
  env_Acacia-H-21-_jpg.rf.671ff77f7b9c3508b329fa6d5820c9a4.jpg: [1.16]
  white_bg_Acacia-Z-159-_jpg.rf.05d5c6dcda22ed9ff794020fd25877e9.jpg: [1.12, 1.16]
  white_bg_Acacia-Z-133-_jpg.rf.a5f7a19138f94e38887579865c4e1801.jpg: [1.12]
  white_bg_Acacia-Z-98-_jpg.rf.055e3c5ce82e87cff44d4b570531ec25.jpg: [1.08]
  white_bg_Acacia-Z-144-_jpg.rf.d13d27b2ae765b17f8c5f4bbad006d8e.jpg: [1.12]
  env_Acacia-P-20-_jpg.rf.826e8110e7fcdba1eb616bc256d5ab84.jpg: [1.06]
  white_bg_Acacia-Z-6-